# Эксперименты с обучением

Все модели — PP-LCNet разной ширины. Логит антисимметричный: $z(x) = \tfrac12[g(x) - g(Rx)]$.

| код | что проверяем |
|---|---|
| E0 | PP-LCNet x0.25 / x1.0 без обучения: как есть и с антисимметричным TTA |
| E1, E2 | дообучение x0.25 / x1.0 на всех трёх датасетах |
| E1b | прирост от головы или от данных: x0.25 с головой против обычного дообучения + TTA |
| E3 | вклад каждого датасета: x0.25 без одного из них |
| E4 | дистилляция x1.0 → x0.25 |
| Q | варианты INT8-квантизации обученных E1/E2: качество и скорость на CPU |

Что запускать, задаёт `Config.run`. Модель выбираем по val, отчёт — на held-out test.
Данные: RusTitW (реальные, RU+EN), TextOCR (реальные, EN), Bilingual (синтетика, RU+EN).

In [ ]:
%pip install -q onnx onnxruntime

In [ ]:
import copy
import io
import json
import os
import random
import re
import time
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass, field
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import onnxruntime as ort
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import Image as HFImage
from datasets import load_dataset
from huggingface_hub import hf_hub_download
from onnxruntime.quantization import (CalibrationDataReader, CalibrationMethod, QuantFormat,
                                      QuantType, quantize_static)
from onnxruntime.quantization.shape_inference import quant_pre_process
from PIL import Image, ImageFilter, ImageOps
from safetensors.torch import load_file
from scipy.optimize import minimize_scalar
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from tqdm.auto import tqdm


@dataclass(frozen=True)
class Config:
    rustitw: Path = Path('/kaggle/input/datasets/hardtype/'
                         'rustitw-russian-language-visual-text-recognition')
    work: Path = Path('/kaggle/working')
    crops: Path = Path('/tmp/rustitw_crops')
    hf_token_secret: str | None = None

    run: tuple = ('E0', 'E1', 'E1b', 'E2', 'E3', 'E4', 'Q')
    e1_checkpoint: Path | None = None  # готовые веса E1/E2: E4 и Q без переобучения
    e2_checkpoint: Path | None = None

    textocr_repo: str = 'MiXaiLL76/TextOCR_OCR'
    textocr_train_shards: int = 2
    textocr_n_train: int = 30_000
    textocr_n_val: int = 3000
    bilingual_repo: str = 'tehnik-tehnolog/bilingual-ocr-ru-en-synthetic'
    bilingual_n_train: int = 30_000  # из официального val; held-out — официальный test
    bilingual_n_val: int = 3000
    bilingual_n_test: int = 10_000
    rustitw_train_photos: int = 3000
    rustitw_val_frac: float = 0.1

    textline_repos: dict = field(default_factory=lambda: {
        0.25: 'PaddlePaddle/PP-LCNet_x0_25_textline_ori_safetensors',
        1.0: 'PaddlePaddle/PP-LCNet_x1_0_textline_ori_safetensors',
    })

    seed: int = 0
    img_h: int = 80
    img_w: int = 160
    epochs: int = 6
    epoch_size: int = 120_000
    batch_size: int = 256
    lr: dict = field(default_factory=lambda: {0.25: 2e-3, 1.0: 7e-4})
    weight_decay: float = 1e-4
    warmup_frac: float = 0.05
    distill_alpha: float = 0.5
    quant_calib_size: int = 512
    quant_variants: dict = field(default_factory=lambda: {
        'qdq minmax per-channel': dict(fmt='qdq', method='minmax', per_channel=True),
        'qdq percentile per-channel': dict(fmt='qdq', method='percentile', per_channel=True),
    })
    num_workers: int = os.cpu_count()
    crop_threads: int = 64


CFG = Config()
DEVICE = 'cuda'
SOURCES = ('rustitw', 'textocr', 'bilingual')
(CFG.work / 'models').mkdir(parents=True, exist_ok=True)

random.seed(CFG.seed)
np.random.seed(CFG.seed)
torch.manual_seed(CFG.seed)

if CFG.hf_token_secret:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret(CFG.hf_token_secret)

## Модель

PP-LCNet на PyTorch с именами параметров как в safetensors PaddleOCR, старт — textline-веса.

In [ ]:
BLOCKS = [
    [(3, 16, 32, 1, False)],
    [(3, 32, 64, (2, 1), False), (3, 64, 64, 1, False)],
    [(3, 64, 128, (2, 1), False), (3, 128, 128, 1, False)],
    [(3, 128, 256, (2, 1), False)] + [(5, 256, 256, 1, False)] * 5,
    [(5, 256, 512, (2, 1), True), (5, 512, 512, 1, True)],
]


def make_div(v, d=8):
    n = max(d, int(v + d / 2) // d * d)
    return n + d if n < 0.9 * v else n


class ConvBNAct(nn.Module):
    def __init__(self, ci, co, k, s, groups=1):
        super().__init__()
        self.convolution = nn.Conv2d(ci, co, k, s, (k - 1) // 2, groups=groups, bias=False)
        self.normalization = nn.BatchNorm2d(co)
        self.activation = nn.Hardswish()

    def forward(self, x):
        return self.activation(self.normalization(self.convolution(x)))


class SE(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.convolutions = nn.ModuleList([
            nn.Conv2d(c, c // 4, 1), nn.ReLU(), nn.Conv2d(c // 4, c, 1), nn.Hardsigmoid(),
        ])

    def forward(self, x):
        s = self.avg_pool(x)
        for layer in self.convolutions:
            s = layer(s)
        return x * s


class DSConv(nn.Module):
    def __init__(self, ci, co, k, s, se):
        super().__init__()
        self.depthwise_convolution = ConvBNAct(ci, ci, k, s, groups=ci)
        self.squeeze_excitation_module = SE(ci) if se else nn.Identity()
        self.pointwise_convolution = ConvBNAct(ci, co, 1, 1)

    def forward(self, x):
        x = self.squeeze_excitation_module(self.depthwise_convolution(x))
        return self.pointwise_convolution(x)


class Block(nn.Module):
    def __init__(self, layers):
        super().__init__()
        self.layers = nn.ModuleList(layers)

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x


class Encoder(nn.Module):
    def __init__(self, scale):
        super().__init__()
        c = lambda v: make_div(v * scale)
        self.convolution = ConvBNAct(3, c(16), 3, 2)
        self.blocks = nn.ModuleList([
            Block([DSConv(c(ci), c(co), k, s, se) for k, ci, co, s, se in block])
            for block in BLOCKS
        ])

    def forward(self, x):
        x = self.convolution(x)
        for block in self.blocks:
            x = block(x)
        return x


class PPLCNet(nn.Module):
    def __init__(self, scale):
        super().__init__()
        self.encoder = Encoder(scale)
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.last_convolution = nn.Conv2d(make_div(512 * scale), 1280, 1, bias=False)
        self.act_fn = nn.Hardswish()
        self.dropout = nn.Dropout(0.2)
        self.head = nn.Linear(1280, 2)

    def forward(self, x):
        x = self.act_fn(self.last_convolution(self.avg_pool(self.encoder(x))))
        return self.head(self.dropout(x.flatten(1)))


class Orient(nn.Module):
    """antisym=True: z(x) = ½[g(x) - g(Rx)]; antisym=False: z(x) = g(x)."""

    def __init__(self, backbone, antisym=True):
        super().__init__()
        self.backbone, self.antisym = backbone, antisym

    def g(self, x):
        out = self.backbone(x)
        return out[:, 1] - out[:, 0]

    def forward(self, x):
        if not self.antisym:
            return self.g(x)
        g = self.g(torch.cat([x, torch.flip(x, [2, 3])])).reshape(2, -1)  # ONNX: без len(x)
        return 0.5 * (g[0] - g[1])


def textline_backbone(scale):
    model = PPLCNet(scale)
    state = load_file(hf_hub_download(CFG.textline_repos[scale], 'model.safetensors'))
    state['head.weight'] = state['head.weight'] * 0.8  # Paddle downscale_in_infer
    model.load_state_dict(state)
    return model

## Данные

* **RusTitW** `train/real`: 10% фото в val, из остальных 3000 фото в train;
  `test/real` — held-out. Рамки один раз вырезаются на локальный диск.
* **TextOCR**: 2 шарда train (30k train + 3k val), `test` — held-out.
* **Bilingual**: train 30k и val 3k из официального val, held-out — 10k из официального test.

`clean`: `w >= h` и есть хотя бы одна буква. В обучение идут кропы с `w >= h` и непустым текстом.

In [ ]:
def rustitw_boxes(split):
    info = pd.read_csv(CFG.rustitw / split / 'real' / 'info.csv')
    rows = []
    for name, w, h, boxes in info[['image_name', 'width', 'height', 'box_and_label']].values:
        for i, b in enumerate(json.loads(boxes)[0]):
            if b['left'] + b['width'] > 1.02 or b['top'] + b['height'] > 1.02:
                continue
            x0, y0 = round(b['left'] * w), round(b['top'] * h)
            x1 = min(w, round((b['left'] + b['width']) * w))
            y1 = min(h, round((b['top'] + b['height']) * h))
            if x1 - x0 < 8 or y1 - y0 < 8:
                continue
            text = b.get('label', '').replace('\\/n', '\n')
            rows.append((name, i, (x0, y0, x1, y1), text))
    return pd.DataFrame(rows, columns=['photo', 'box_id', 'box', 'text'])


def crop_photo(split, photo, group):
    src = CFG.rustitw / split / 'real' / 'images' / photo
    paths = [CFG.crops / split / f'{Path(photo).stem}_{i}.png' for i in group.box_id]
    if not all(p.exists() for p in paths):
        with Image.open(src) as im:
            im = im.convert('RGB')
            for box, path in zip(group.box, paths):
                im.crop(box).save(path)
    return paths


def rustitw_frame(split, photos=None):
    df = rustitw_boxes(split)
    if photos is not None:
        df = df[df.photo.isin(photos)]
    (CFG.crops / split).mkdir(parents=True, exist_ok=True)
    groups = list(df.groupby('photo', sort=False))
    with ThreadPoolExecutor(CFG.crop_threads) as ex:
        jobs = ex.map(lambda item: crop_photo(split, *item), groups)
        paths = list(tqdm(jobs, total=len(groups), desc=f'RusTitW {split}: crops'))
    df = pd.concat([g for _, g in groups]).assign(ref=[str(p) for ps in paths for p in ps])
    w, h = zip(*[(b[2] - b[0], b[3] - b[1]) for b in df.box])
    return df.assign(src='rustitw', w=w, h=h)[['src', 'photo', 'text', 'w', 'h', 'ref']]


def hf_frame(ds, src, key, idx):
    raw = ds.cast_column('image', HFImage(decode=False))
    rows = []
    for i in tqdm(idx, desc=f'{src} {key}: sizes'):
        r = raw[int(i)]
        with Image.open(io.BytesIO(r['image']['bytes'])) as im:
            rows.append((src, '', r['text'], *im.size, f'{src}/{key}:{i}'))
    return pd.DataFrame(rows, columns=['src', 'photo', 'text', 'w', 'h', 'ref'])


def add_flags(df):
    text = df.text.astype(str)
    cyr = text.str.contains('[А-Яа-яЁё]')
    lat = text.str.contains('[A-Za-z]')
    script = np.select([cyr & lat, cyr, lat], ['mixed', 'cyrillic', 'latin'], 'no_letters')
    return df.assign(script=script, n_lines=text.str.count('\n') + 1,
                     clean=(df.w >= df.h) & (script != 'no_letters'))


def load_hf(repo, files, split):
    return load_dataset(repo, data_files={split: files}, split=split,
                        verification_mode='no_checks')


rng = np.random.default_rng(CFG.seed)
photos = rng.permutation(rustitw_boxes('train').photo.unique())
n_val = int(len(photos) * CFG.rustitw_val_frac)
rt_val = rustitw_frame('train', photos[:n_val])
rt_train = rustitw_frame('train', photos[n_val:n_val + CFG.rustitw_train_photos])
rt_test = rustitw_frame('test')

TEXTOCR_SHARDS = [f'data/train-{i:05d}-of-00005.parquet' for i in range(CFG.textocr_train_shards)]
HF = {
    'textocr/train': load_hf(CFG.textocr_repo, TEXTOCR_SHARDS, 'train'),
    'textocr/test': load_hf(CFG.textocr_repo, 'data/test-00000-of-00001.parquet', 'test'),
    'bilingual/val': load_hf(CFG.bilingual_repo, 'data/val-00000-of-00001.parquet', 'val'),
    'bilingual/test': load_hf(CFG.bilingual_repo, 'data/test-00000-of-00001.parquet', 'test'),
}
to_idx = rng.permutation(len(HF['textocr/train']))
to_train = hf_frame(HF['textocr/train'], 'textocr', 'train', to_idx[:CFG.textocr_n_train])
to_val = hf_frame(HF['textocr/train'], 'textocr', 'train',
                  to_idx[CFG.textocr_n_train:CFG.textocr_n_train + CFG.textocr_n_val])
to_test = hf_frame(HF['textocr/test'], 'textocr', 'test', range(len(HF['textocr/test'])))

bi_idx = rng.permutation(len(HF['bilingual/val']))  # строки отсортированы по языку
bi_train = hf_frame(HF['bilingual/val'], 'bilingual', 'val', bi_idx[:CFG.bilingual_n_train])
bi_val = hf_frame(HF['bilingual/val'], 'bilingual', 'val',
                  bi_idx[CFG.bilingual_n_train:CFG.bilingual_n_train + CFG.bilingual_n_val])
bi_test = hf_frame(HF['bilingual/test'], 'bilingual', 'test',
                   rng.permutation(len(HF['bilingual/test']))[:CFG.bilingual_n_test])

SPLITS = {
    'train': add_flags(pd.concat([rt_train, to_train, bi_train], ignore_index=True)),
    'val': add_flags(pd.concat([rt_val, to_val, bi_val], ignore_index=True)),
    'test': add_flags(pd.concat([rt_test, to_test, bi_test], ignore_index=True)),
}
pd.concat({k: v.groupby('src').agg(n=('clean', 'size'), clean=('clean', 'mean'))
           for k, v in SPLITS.items()}).round(3)

## Аугментации и загрузчик

Кроп сначала уменьшается до ~2× целевого размера, затем: джиттер рамки, наклон,
уменьшение до 8–32 px, блюр, окклюдер, засветка, яркость/контраст, шум, инверсия,
grayscale, JPEG. Поворот на 180° делается в батче на GPU.

In [ ]:
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)


def load_image(ref):
    if ref.startswith('/'):
        with Image.open(ref) as im:
            return im.convert('RGB')
    key, idx = ref.split(':')
    return HF[key][int(idx)]['image'].convert('RGB')


def to_tensor(img):
    x = np.asarray(img.resize((CFG.img_w, CFG.img_h), Image.BILINEAR), np.float32) / 255
    return (torch.from_numpy(x).permute(2, 0, 1) - MEAN) / STD


def shrink(img):
    w, h = img.size
    s = min(1.0, 2 * CFG.img_h / h, 4 * CFG.img_w / w)
    return img.resize((max(8, round(w * s)), max(8, round(h * s)))) if s < 1 else img


def mean_color(img):
    return tuple(np.asarray(img).reshape(-1, 3).mean(0).astype(int))


def augment(img):
    img = shrink(img)
    w, h = img.size
    if random.random() < 0.7:
        d = [int(random.uniform(-0.12, 0.12) * h) for _ in range(4)]
        img = ImageOps.expand(img, tuple(max(0, -v) for v in d), fill=mean_color(img))
        c = [max(0, v) for v in d]
        img = img.crop((c[0], c[1], max(c[0] + 4, img.width - c[2]),
                        max(c[1] + 4, img.height - c[3])))
    if random.random() < 0.3:
        img = img.rotate(random.uniform(-4, 4), Image.BILINEAR, fillcolor=mean_color(img))
    if random.random() < 0.5:
        th = random.randint(8, 32)
        if img.height > th:
            img = img.resize((max(4, round(img.width * th / img.height)), th), Image.BILINEAR)
    if random.random() < 0.25:
        img = img.filter(ImageFilter.GaussianBlur(random.uniform(0.3, 1.5)))

    a = np.asarray(img, np.float32)
    hh, ww = a.shape[:2]
    if random.random() < 0.2:
        yy, xx = np.mgrid[:hh, :ww]
        cy, cx = random.choice([0, hh]) + random.uniform(-0.3, 0.3) * hh, random.uniform(0, ww)
        ry, rx = random.uniform(0.4, 1.2) * hh, random.uniform(0.1, 0.4) * ww
        a[((yy - cy) / ry) ** 2 + ((xx - cx) / rx) ** 2 < 1] = np.random.randint(0, 256, 3)
    if random.random() < 0.2:
        gx = np.linspace(-1, 1, ww)[None, :] * random.uniform(-70, 70)
        gy = np.linspace(-1, 1, hh)[:, None] * random.uniform(-50, 50)
        a = a + (gx + gy)[..., None]
    if random.random() < 0.5:
        a = (a - a.mean()) * random.uniform(0.6, 1.4) + a.mean() + random.uniform(-30, 30)
    if random.random() < 0.3:
        a = a + np.random.normal(0, random.uniform(2, 12), a.shape)
    img = Image.fromarray(np.clip(a, 0, 255).astype(np.uint8))

    if random.random() < 0.2:
        img = ImageOps.invert(img)
    if random.random() < 0.15:
        img = ImageOps.grayscale(img).convert('RGB')
    if random.random() < 0.4:
        buf = io.BytesIO()
        img.save(buf, 'JPEG', quality=random.randint(20, 90))
        img = Image.open(buf).convert('RGB')
    return img


class CropDataset(Dataset):
    def __init__(self, refs, train):
        self.refs, self.train = list(refs), train

    def __len__(self):
        return len(self.refs)

    def __getitem__(self, i):
        img = load_image(self.refs[i])
        return to_tensor(augment(img) if self.train else img)


def train_loader(sources, epoch_size):
    df = SPLITS['train']
    df = df[df.src.isin(sources) & (df.w >= df.h) & (df.text.str.strip() != '')]
    weights = 1.0 / df.src.map(df.src.value_counts()).to_numpy()
    sampler = WeightedRandomSampler(torch.tensor(weights), epoch_size,
                                    generator=torch.Generator().manual_seed(CFG.seed))
    return DataLoader(CropDataset(df.ref, train=True), CFG.batch_size, sampler=sampler,
                      num_workers=CFG.num_workers, pin_memory=True, drop_last=True)

## Оценка

Режимы: `raw` — логиты на $x$ и $Rx$ как есть; `tta` — их антисимметричная часть
$\tfrac12[z(x) - z(Rx)]$; `model` — модель уже антисимметрична, считаем только $z(x)$,
а $z(Rx) = -z(x)$.
Температура — по Brier на val (clean), отчёт — на held-out test.

In [ ]:
sigmoid = lambda z: 1 / (1 + np.exp(-z))
RESULTS = {}


@torch.no_grad()
def torch_logits(model):
    model.eval()
    return lambda x: model(x.to(DEVICE)).float().cpu().numpy()


def onnx_logits(path, threads=None):
    opts = ort.SessionOptions()
    if threads:
        opts.intra_op_num_threads = threads
    session = ort.InferenceSession(str(path), opts, providers=['CPUExecutionProvider'])
    return lambda x: session.run(None, {'x': x.numpy()})[0]


def predict(logits_fn, df, mode):
    loader = DataLoader(CropDataset(df.ref, train=False), 512, num_workers=CFG.num_workers)
    up, rot = [], []
    with torch.autocast('cuda'):
        for x in tqdm(loader, desc='predict', leave=False):
            up.append(logits_fn(x))
            if mode != 'model':
                rot.append(logits_fn(torch.flip(x, [2, 3])))
    up = np.concatenate(up)
    return up, -up if mode == 'model' else np.concatenate(rot)


def pairs(up, rot, mode):
    if mode == 'tta':
        up = 0.5 * (up - rot)
        rot = -up
    return np.r_[up, rot], np.r_[np.zeros(len(up)), np.ones(len(rot))]


def ece(y, p, bins=15):
    conf, correct = np.maximum(p, 1 - p), (p > 0.5) == y
    idx = np.clip(np.digitize(conf, np.linspace(0.5, 1, bins + 1)) - 1, 0, bins - 1)
    return sum(np.mean(idx == b) * abs(correct[idx == b].mean() - conf[idx == b].mean())
               for b in range(bins) if np.any(idx == b))


def metrics(z, y, T):
    p = sigmoid(z / T)
    return {'1-Brier': 1 - np.mean((p - y) ** 2), 'AUC': roc_auc_score(y, z), 'ECE': ece(y, p)}


def slices(df):
    rt, to, bi, cl = df.src == 'rustitw', df.src == 'textocr', df.src == 'bilingual', df.clean
    cyr, lat = df.script == 'cyrillic', df.script == 'latin'
    return {
        'RusTitW clean': rt & cl,
        'RusTitW cyr': rt & cl & cyr,
        'RusTitW cyr 1-line': rt & cl & cyr & (df.n_lines == 1),
        'RusTitW cyr multi-line': rt & cl & cyr & (df.n_lines > 1),
        'RusTitW lat': rt & cl & lat,
        'RusTitW vertical': rt & (df.w < df.h),
        'TextOCR clean': to & cl,
        'TextOCR all': to,
        'Bilingual': bi,
    }


SLICES = {split: {k: m.to_numpy() for k, m in slices(SPLITS[split]).items()}
          for split in ('val', 'test')}
VAL_REAL = SLICES['val']['RusTitW clean'] | SLICES['val']['TextOCR clean']


def evaluate(name, logits_fn, mode, **extra):
    val, test = predict(logits_fn, SPLITS['val'], mode), predict(logits_fn, SPLITS['test'], mode)
    z, y = pairs(val[0][VAL_REAL], val[1][VAL_REAL], mode)
    T = minimize_scalar(lambda t: np.mean((sigmoid(z / t) - y) ** 2),
                        bounds=(0.05, 20), method='bounded').x
    RESULTS[name] = dict(mode=mode, T=T, val=val, test=test, **extra)
    print(f'{name:36s} T={T:5.2f}  val-real 1-Brier {val_score(name):.4f}')


def val_score(name, T=None):
    r = RESULTS[name]
    return np.mean([metrics(*pairs(r['val'][0][m], r['val'][1][m], r['mode']),
                            r['T'] if T is None else T)['1-Brier']
                    for m in (SLICES['val']['RusTitW clean'], SLICES['val']['TextOCR clean'])])


def table(split='test', names=None, metric='1-Brier'):
    rows = {}
    for name in names or RESULTS:
        r = RESULTS[name]
        up, rot = r[split]
        rows[name] = {k: metrics(*pairs(up[m], rot[m], r['mode']), r['T'])[metric]
                      for k, m in SLICES[split].items()}
    return pd.DataFrame(rows).T

## Обучение

BCE на логите $z$, AdamW, one-cycle LR, AMP. После каждой эпохи смотрим val-real
(среднее 1-Brier по RusTitW clean и TextOCR clean при $T=1$), сохраняем лучшую эпоху.
Для дистилляции к лоссу добавляется $\alpha\cdot\mathrm{MSE}(z, z_{\text{teacher}})$.

In [ ]:
def val_real_raw(model):
    df = SPLITS['val'][VAL_REAL]
    mode = 'model' if model.antisym else 'tta'
    up, rot = predict(torch_logits(model), df, mode)
    s = slices(df)
    return np.mean([metrics(*pairs(up[m.to_numpy()], rot[m.to_numpy()], mode), 1.0)['1-Brier']
                    for m in (s['RusTitW clean'], s['TextOCR clean'])])


def train(name, scale, sources=SOURCES, antisym=True, teacher=None,
          epochs=CFG.epochs, epoch_size=CFG.epoch_size):
    model = Orient(textline_backbone(scale), antisym).to(DEVICE)
    loader = train_loader(sources, epoch_size)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG.lr[scale], weight_decay=CFG.weight_decay)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=CFG.lr[scale],
                                                pct_start=CFG.warmup_frac,
                                                total_steps=epochs * len(loader))
    scaler = torch.amp.GradScaler()
    best, best_state = -1.0, None
    for epoch in range(1, epochs + 1):
        model.train()
        t0, losses = time.time(), []
        for x in tqdm(loader, desc=f'{name} {epoch}/{epochs}', leave=False):
            x = x.to(DEVICE, non_blocking=True)
            y = (torch.rand(len(x), device=DEVICE) < 0.5).float()
            x = torch.where(y[:, None, None, None] > 0, torch.flip(x, [2, 3]), x)
            with torch.autocast('cuda'):
                z = model(x).float()
                loss = F.binary_cross_entropy_with_logits(z, y)
                if teacher is not None:
                    with torch.no_grad():
                        z_t = teacher(x).float()
                    loss = (1 - CFG.distill_alpha) * loss + CFG.distill_alpha * F.mse_loss(z, z_t)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
            losses.append(loss.item())
        score = val_real_raw(model)
        print(f'  epoch {epoch}: loss {np.mean(losses):.4f}  val-real {score:.4f}  '
              f'({time.time() - t0:.0f}s)')
        if score > best:
            best, best_state = score, copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    ckpt = CFG.work / 'models' / f"{re.sub('[^A-Za-z0-9.]+', '_', name)}.pt"
    torch.save(best_state, ckpt)
    evaluate(name, torch_logits(model), 'model' if antisym else 'tta', scale=scale, ckpt=ckpt)
    return model


def load_trained(name, scale, ckpt):
    model = Orient(PPLCNet(scale)).to(DEVICE)
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    evaluate(name, torch_logits(model), 'model', scale=scale, ckpt=ckpt)
    return model

## E0: бейзлайны без обучения

In [ ]:
if 'E0' in CFG.run:
    for scale in (0.25, 1.0):
        model = Orient(textline_backbone(scale), antisym=False).to(DEVICE)
        evaluate(f'E0 x{scale} raw', torch_logits(model), 'raw', scale=scale)
        evaluate(f'E0 x{scale} +TTA', torch_logits(model), 'tta', scale=scale)
    display(table('val').round(4))

## E1 / E2: дообучение x0.25 и x1.0

In [ ]:
MODELS = {}
if 'E1' in CFG.run:
    MODELS['E1'] = train('E1 x0.25', 0.25)
elif CFG.e1_checkpoint is not None:
    MODELS['E1'] = load_trained('E1 x0.25', 0.25, CFG.e1_checkpoint)
if 'E2' in CFG.run:
    MODELS['E2'] = train('E2 x1.0', 1.0)
elif CFG.e2_checkpoint is not None:
    MODELS['E2'] = load_trained('E2 x1.0', 1.0, CFG.e2_checkpoint)

## E1b: прирост даёт голова или данные

Одинаковый бюджет (3 × 30k), x0.25. Голова в обучении против обычного дообучения;
на инференсе у обоих антисимметричный логит (у второго — через TTA).

In [ ]:
if 'E1b' in CFG.run:
    budget = dict(epochs=3, epoch_size=30_000)
    train('E1b x0.25 head', 0.25, **budget)
    plain = train('E1b x0.25 plain', 0.25, antisym=False, **budget)
    evaluate('E1b x0.25 plain raw', torch_logits(plain), 'raw', scale=0.25)
    display(table(names=[n for n in RESULTS if n.startswith(('E0 x0.25 +', 'E1b'))]).round(4))

## E3: вклад каждого датасета (x0.25 без одного источника)

In [ ]:
if 'E3' in CFG.run:
    for drop in SOURCES:
        train(f'E3 x0.25 no-{drop}', 0.25, sources=[s for s in SOURCES if s != drop], epochs=3)
    display(table(names=[n for n in RESULTS if n.startswith(('E1 ', 'E3'))]).round(4))

## E4: дистилляция x1.0 → x0.25 (без Bilingual)

In [ ]:
if 'E4' in CFG.run:
    teacher = MODELS['E2'].eval()
    train('E4 x0.25 distill', 0.25, sources=('rustitw', 'textocr'), teacher=teacher,
          epochs=4, epoch_size=60_000)

## Q: варианты INT8-квантизации

Обученные E1 (x0.25) и E2 (x1.0) вместе с антисимметричной головой экспортируем в ONNX,
прогоняем `quant_pre_process` (вывод форм и слияние узлов) и квантуем статически
(веса INT8, активации UINT8) по вариантам из `Config.quant_variants`:

* формат: QDQ (узлы Quantize/Dequantize вокруг обычных операций) или QOperator
  (готовые INT8-операции);
* калибровка диапазонов активаций: MinMax, Percentile (обрезает выбросы), Entropy;
* масштаб весов: по каналам или один на весь тензор.

Калибровка — на `quant_calib_size` кропах val. Динамическая квантизация не подходит:
в onnxruntime для CPU нет INT8-свёртки `ConvInteger`.

Локальная проверка на весах PaddleOCR (CPU Mac) показала, что пост-квантизация ломает модель
при любых настройках: знак логита совпадает с FP32 лишь в ~60% (x0.25) и ~80% (x1.0)
случаев, хотя скорость растёт в ~2.5 раза. Не помогают ни percentile/entropy-калибровка,
ни масштаб по тензору, ни исключение первой свёртки и головы; формат QOperator для x1.0
падает с ошибкой. Здесь оставлены два варианта, чтобы зафиксировать это на обученных
моделях и на полном held-out.
Сравниваем с FP32 ONNX по качеству на held-out и по скорости на CPU (1 поток, batch 1).

In [ ]:
METHODS = {'minmax': CalibrationMethod.MinMax, 'percentile': CalibrationMethod.Percentile,
           'entropy': CalibrationMethod.Entropy}
FORMATS = {'qdq': QuantFormat.QDQ, 'qoperator': QuantFormat.QOperator}


class ValReader(CalibrationDataReader):
    def __init__(self, n):
        refs = SPLITS['val'].ref.sample(n, random_state=CFG.seed)
        self.batches = iter([{'x': to_tensor(load_image(r))[None].numpy()} for r in refs])

    def get_next(self):
        return next(self.batches, None)


def export_onnx(model, path):
    torch.onnx.export(copy.deepcopy(model).cpu().eval(),
                      torch.randn(1, 3, CFG.img_h, CFG.img_w), path,
                      input_names=['x'], output_names=['logit_180'], opset_version=17,
                      dynamic_axes={'x': {0: 'n'}, 'logit_180': {0: 'n'}}, dynamo=False)


def quantize(src, dst, fmt, method, per_channel):
    quantize_static(src, dst, ValReader(CFG.quant_calib_size), quant_format=FORMATS[fmt],
                    calibrate_method=METHODS[method], per_channel=per_channel,
                    weight_type=QuantType.QInt8, activation_type=QuantType.QUInt8)


def latency_ms(path, reps=300):
    run = onnx_logits(path, threads=1)
    x = torch.randn(1, 3, CFG.img_h, CFG.img_w)
    for _ in range(30):
        run(x)
    t0 = time.perf_counter()
    for _ in range(reps):
        run(x)
    return (time.perf_counter() - t0) / reps * 1e3


if 'Q' in CFG.run:
    speed = []
    for key, model in MODELS.items():
        raw = CFG.work / 'models' / f'{key}.onnx'
        fp32 = CFG.work / 'models' / f'{key}.pre.onnx'
        export_onnx(model, raw)
        quant_pre_process(str(raw), str(fp32))
        variants = {'fp32': fp32}
        for tag, v in CFG.quant_variants.items():
            variants[tag] = CFG.work / 'models' / f"{key}.{tag.replace(' ', '_')}.onnx"
            quantize(fp32, variants[tag], **v)
        for tag, path in variants.items():
            evaluate(f'Q {key} {tag}', onnx_logits(path), 'model')
            speed.append({'model': key, 'variant': tag, 'MB': path.stat().st_size / 1e6,
                          'ms': latency_ms(path)})
    speed = pd.DataFrame(speed)
    quality = table(names=[n for n in RESULTS if n.startswith('Q ')])
    speed['1-Brier RusTitW'] = quality['RusTitW clean'].to_numpy()
    speed['1-Brier TextOCR'] = quality['TextOCR clean'].to_numpy()
    display(speed.round(4))

## Выбор по val и отчёт на held-out test

In [ ]:
val_rank = pd.Series({name: val_score(name) for name in RESULTS}).sort_values(ascending=False)
display(val_rank.rename('val-real 1-Brier').round(4).to_frame())
print('1-Brier, held-out test')
display(table('test').round(4))
print('AUC, held-out test')
display(table('test', metric='AUC').round(4))
print('ECE, held-out test')
display(table('test', metric='ECE').round(4))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5), sharey=True)
for ax, sl in zip(axes, ['RusTitW cyr', 'RusTitW lat', 'TextOCR clean']):
    m = SLICES['test'][sl]
    for name in val_rank.index[:4]:
        r = RESULTS[name]
        z, y = pairs(r['test'][0][m], r['test'][1][m], r['mode'])
        p = sigmoid(z / r['T'])
        bins = pd.qcut(p, 12, labels=False, duplicates='drop')
        ax.plot(pd.Series(p).groupby(bins).mean(), pd.Series(y).groupby(bins).mean(), 'o-',
                ms=3, label=name)
    ax.plot([0, 1], [0, 1], 'k:')
    ax.set(title=sl, xlabel='p_180')
    ax.grid(alpha=0.3)
axes[0].set_ylabel('доля перевёрнутых')
axes[-1].legend(fontsize=7)
plt.tight_layout()
plt.show()

In [ ]:
summary = {
    'config': {k: str(v) for k, v in vars(CFG).items()},
    'val_rank': val_rank.round(5).to_dict(),
    'test': table('test').round(5).to_dict(orient='index'),
    'T': {name: r['T'] for name, r in RESULTS.items()},
}
(CFG.work / 'experiments.json').write_text(json.dumps(summary, indent=1, ensure_ascii=False))